# TC2x2 por registradores e por coluna

Este notebook implementa a convolução rápida **TC2x2** para um tile de entrada \(4\times4\), kernel \(3\times3\) e saída \(2\times2\).

O fluxo matemático é:

$$
D = C^{T} d C
$$

$$
G = (QB)g(QB)^{T}
$$

$$
S = D \odot G
$$

$$
s = A^{T} S A
$$

A implementação principal é feita em estilo **registrador por registrador**, produzindo \(D\) uma coluna por vez e sem materializar \(D\) nem \(S\) inteiros no datapath principal.


## Matrizes usadas

$$
\begin{aligned}
C &= \begin{bmatrix}
-1 & 0 & 0 & 0 \\
0 & 1 & -1 & -1 \\
1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{aligned}
$$

$$
\begin{aligned}
A &= \begin{bmatrix}
1 & 0 \\
1 & 1 \\
1 & -1 \\
0 & 1
\end{bmatrix}
\end{aligned}
$$

$$
\begin{aligned}
QB &= \begin{bmatrix}
-1 & 0 & 0 \\
\frac{1}{2} & \frac{1}{2} & \frac{1}{2} \\
\frac{1}{2} & -\frac{1}{2} & \frac{1}{2} \\
0 & 0 & 1
\end{bmatrix}
\end{aligned}
$$

Nesta formulação, \(QB\) já incorpora o fator de escala \(Q\) aplicado às linhas de \(B\).


In [1]:
from fractions import Fraction as F
import numpy as np


## 1. Referência matricial

Esta célula define as matrizes apenas para verificação.


In [2]:
C = np.array([
    [-1, 0,  0,  0],
    [ 0, 1, -1, -1],
    [ 1, 1,  1,  0],
    [ 0, 0,  0,  1],
], dtype=object)

A = np.array([
    [1,  0],
    [1,  1],
    [1, -1],
    [0,  1],
], dtype=object)

QB = np.array([
    [F(-1),   F(0),    F(0)],
    [F(1,2), F(1,2),  F(1,2)],
    [F(1,2), F(-1,2), F(1,2)],
    [F(0),   F(0),    F(1)],
], dtype=object)

print("C shape: ", C.shape)
print("A shape: ", A.shape)
print("QB shape:", QB.shape)


C shape:  (4, 4)
A shape:  (4, 2)
QB shape: (4, 3)


## 2. Entradas editáveis

Os valores abaixo reproduzem o exemplo usado na conversa:

$$
\begin{aligned}
d &= \begin{bmatrix}
0 & 1 & 2 & 3 \\
4 & 5 & 6 & 7 \\
8 & 9 & 10 & 11 \\
12 & 13 & 14 & 15
\end{bmatrix}
\end{aligned}
$$

$$
\begin{aligned}
g &= \begin{bmatrix}
0 & 1 & 2 \\
3 & 4 & 5 \\
6 & 7 & 8
\end{bmatrix}
\end{aligned}
$$


In [3]:
d_np = np.array([
    [ 0,  1,  2,  3],
    [ 4,  5,  6,  7],
    [ 8,  9, 10, 11],
    [12, 13, 14, 15],
], dtype=object)

g_np = np.array([
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],
], dtype=object)

D_ref = C.T @ d_np @ C
G_ref = QB @ g_np @ QB.T
S_ref = D_ref * G_ref
s_ref = A.T @ S_ref @ A

print("D_ref =")
print(D_ref)
print()
print("G_ref =")
print(G_ref)
print()
print("S_ref =")
print(S_ref)
print()
print("s_ref =")
print(s_ref)


D_ref =
[[0 16 0 0]
 [4 30 2 4]
 [0 8 0 0]
 [0 16 0 0]]

G_ref =
[[Fraction(0, 1) Fraction(-3, 2) Fraction(-1, 2) Fraction(-2, 1)]
 [Fraction(-9, 2) Fraction(9, 1) Fraction(3, 1) Fraction(15, 2)]
 [Fraction(-3, 2) Fraction(3, 1) Fraction(1, 1) Fraction(5, 2)]
 [Fraction(-6, 1) Fraction(21, 2) Fraction(7, 2) Fraction(8, 1)]]

S_ref =
[[Fraction(0, 1) Fraction(-24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(-18, 1) Fraction(270, 1) Fraction(6, 1) Fraction(30, 1)]
 [Fraction(0, 1) Fraction(24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(0, 1) Fraction(168, 1) Fraction(0, 1) Fraction(0, 1)]]

s_ref =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]


## 3. Versão por registradores

Nesta versão, cada variável escalar representa um registrador ou um valor lido de ROM/memória.

### Registradores principais do datapath

- `d00..d33`: tile de entrada \(d[4\times4]\).  
- `acc0..acc3`: coluna atual de \(D\).  
- `y0,y1`: resultado intermediário \(y=A^T S_{:,j}\).  
- `s00..s11`: acumuladores finais da saída \(s[2\times2]\).  

A transformação de pesos \(G=(QB)g(QB)^T\) também é feita de forma explícita. Em uma arquitetura real, \(G\) pode ser pré-computado offline e lido de ROM/SRAM.


In [4]:
# ============================================================
# REGISTRADORES DE ENTRADA d
# Propósito: armazenar o tile de entrada 4x4.
# ============================================================

d00 = F(0)
d01 = F(1)
d02 = F(2)
d03 = F(3)

d10 = F(4)
d11 = F(5)
d12 = F(6)
d13 = F(7)

d20 = F(8)
d21 = F(9)
d22 = F(10)
d23 = F(11)

d30 = F(12)
d31 = F(13)
d32 = F(14)
d33 = F(15)

# ============================================================
# REGISTRADORES DO FILTRO ORIGINAL g
# Propósito: armazenar o kernel 3x3 antes da transformação.
# ============================================================

g00 = F(0)
g01 = F(1)
g02 = F(2)

g10 = F(3)
g11 = F(4)
g12 = F(5)

g20 = F(6)
g21 = F(7)
g22 = F(8)

# ============================================================
# REGISTRADORES DO FILTRO TRANSFORMADO G
# Propósito:
#   armazenar G = (QB) g (QB)^T.
# Se G vier pré-computado de ROM/SRAM, estes podem ser lidos.
# ============================================================

G00 = F(0); G01 = F(0); G02 = F(0); G03 = F(0)
G10 = F(0); G11 = F(0); G12 = F(0); G13 = F(0)
G20 = F(0); G21 = F(0); G22 = F(0); G23 = F(0)
G30 = F(0); G31 = F(0); G32 = F(0); G33 = F(0)

# ============================================================
# REGISTRADORES DA COLUNA ATUAL DE D
# Propósito:
#   acc0 = D[0,j]
#   acc1 = D[1,j]
#   acc2 = D[2,j]
#   acc3 = D[3,j]
# Reutilizados para cada coluna j.
# ============================================================

acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)

# ============================================================
# REGISTRADORES INTERMEDIÁRIOS y
# Propósito:
#   y = A^T @ S[:,j]
#
# Como A^T é 2x4:
#   y0 = S0j + S1j + S2j
#   y1 =       S1j - S2j + S3j
# ============================================================

y0 = F(0)
y1 = F(0)

# ============================================================
# REGISTRADORES DE SAÍDA s
# Propósito:
#   acumular s = A^T S A.
# ============================================================

s00 = F(0)
s01 = F(0)
s10 = F(0)
s11 = F(0)


## 4. Transformação explícita do filtro

Implementa:

$$
G = (QB)g(QB)^{T}
$$

A transformação é feita em duas etapas conceituais:

$$
T = (QB)g
$$

$$
G = T(QB)^{T}
$$

Os registradores `T00..T32` são intermediários da transformação de peso. Como o peso normalmente pode ser pré-computado offline, estes intermediários não precisam fazer parte do datapath online.


In [5]:
# ============================================================
# INTERMEDIÁRIOS T = QB @ g
# ============================================================

T00 = -g00
T01 = -g01
T02 = -g02

T10 = (g00 + g10 + g20) / 2
T11 = (g01 + g11 + g21) / 2
T12 = (g02 + g12 + g22) / 2

T20 = (g00 - g10 + g20) / 2
T21 = (g01 - g11 + g21) / 2
T22 = (g02 - g12 + g22) / 2

T30 = g20
T31 = g21
T32 = g22

# ============================================================
# G = T @ QB.T
# ============================================================

G00 = -T00
G01 = (T00 + T01 + T02) / 2
G02 = (T00 - T01 + T02) / 2
G03 = T02

G10 = -T10
G11 = (T10 + T11 + T12) / 2
G12 = (T10 - T11 + T12) / 2
G13 = T12

G20 = -T20
G21 = (T20 + T21 + T22) / 2
G22 = (T20 - T21 + T22) / 2
G23 = T22

G30 = -T30
G31 = (T30 + T31 + T32) / 2
G32 = (T30 - T31 + T32) / 2
G33 = T32

G_reg = np.array([
    [G00, G01, G02, G03],
    [G10, G11, G12, G13],
    [G20, G21, G22, G23],
    [G30, G31, G32, G33],
], dtype=object)

print("G_reg =")
print(G_reg)
print()
print("G_reg == G_ref?", np.array_equal(G_reg, G_ref))


G_reg =
[[Fraction(0, 1) Fraction(-3, 2) Fraction(-1, 2) Fraction(-2, 1)]
 [Fraction(-9, 2) Fraction(9, 1) Fraction(3, 1) Fraction(15, 2)]
 [Fraction(-3, 2) Fraction(3, 1) Fraction(1, 1) Fraction(5, 2)]
 [Fraction(-6, 1) Fraction(21, 2) Fraction(7, 2) Fraction(8, 1)]]

G_reg == G_ref? True


## 5. Transformada de dados por coluna e inversa acumulada

Esta célula executa o fluxo online principal:

$$
D_{:,j} = C^{T} d C_{:,j}
$$

$$
S_{:,j} = D_{:,j}\odot G_{:,j}
$$

$$
y = A^{T} S_{:,j}
$$

$$
s mathrel{+}= y A_{j,:}
$$

Assim, \(D\) e \(S\) não são materializados inteiros.


In [6]:
# ============================================================
# COLUNA 0 DE D
# C[:,0] = [-1, 0, 1, 0]^T
# A[0,:] = [1, 0]
# ============================================================

acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)

# d[0,:] @ C[:,0] = -d00 + d02
acc0 -= (-d00 + d02)

# d[1,:] @ C[:,0] = -d10 + d12
acc1 += (-d10 + d12)
acc2 -= (-d10 + d12)
acc3 -= (-d10 + d12)

# d[2,:] @ C[:,0] = -d20 + d22
acc0 += (-d20 + d22)
acc1 += (-d20 + d22)
acc2 += (-d20 + d22)

# d[3,:] @ C[:,0] = -d30 + d32
acc3 += (-d30 + d32)

# y = A.T @ (D[:,0] ⊙ G[:,0])
y0 = F(0)
y1 = F(0)

y0 += acc0 * G00
y0 += acc1 * G10
y1 += acc1 * G10
y0 += acc2 * G20
y1 -= acc2 * G20
y1 += acc3 * G30

# s += y @ A[0,:]
s00 += y0
s10 += y1


# ============================================================
# COLUNA 1 DE D
# C[:,1] = [0, 1, 1, 0]^T
# A[1,:] = [1, 1]
# ============================================================

acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)

acc0 -= (d01 + d02)

acc1 += (d11 + d12)
acc2 -= (d11 + d12)
acc3 -= (d11 + d12)

acc0 += (d21 + d22)
acc1 += (d21 + d22)
acc2 += (d21 + d22)

acc3 += (d31 + d32)

y0 = F(0)
y1 = F(0)

y0 += acc0 * G01
y0 += acc1 * G11
y1 += acc1 * G11
y0 += acc2 * G21
y1 -= acc2 * G21
y1 += acc3 * G31

s00 += y0
s01 += y0
s10 += y1
s11 += y1


# ============================================================
# COLUNA 2 DE D
# C[:,2] = [0, -1, 1, 0]^T
# A[2,:] = [1, -1]
# ============================================================

acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)

acc0 -= (-d01 + d02)

acc1 += (-d11 + d12)
acc2 -= (-d11 + d12)
acc3 -= (-d11 + d12)

acc0 += (-d21 + d22)
acc1 += (-d21 + d22)
acc2 += (-d21 + d22)

acc3 += (-d31 + d32)

y0 = F(0)
y1 = F(0)

y0 += acc0 * G02
y0 += acc1 * G12
y1 += acc1 * G12
y0 += acc2 * G22
y1 -= acc2 * G22
y1 += acc3 * G32

s00 += y0
s01 -= y0
s10 += y1
s11 -= y1


# ============================================================
# COLUNA 3 DE D
# C[:,3] = [0, -1, 0, 1]^T
# A[3,:] = [0, 1]
# ============================================================

acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)

acc0 -= (-d01 + d03)

acc1 += (-d11 + d13)
acc2 -= (-d11 + d13)
acc3 -= (-d11 + d13)

acc0 += (-d21 + d23)
acc1 += (-d21 + d23)
acc2 += (-d21 + d23)

acc3 += (-d31 + d33)

y0 = F(0)
y1 = F(0)

y0 += acc0 * G03
y0 += acc1 * G13
y1 += acc1 * G13
y0 += acc2 * G23
y1 -= acc2 * G23
y1 += acc3 * G33

s01 += y0
s11 += y1


s_reg = np.array([
    [s00, s01],
    [s10, s11],
], dtype=object)

print("s_reg =")
print(s_reg)
print()
print("s_reg == s_ref?", np.array_equal(s_reg, s_ref))


s_reg =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]

s_reg == s_ref? True


## 6. Conferência dos intermediários \(D\), \(G\), \(S\)

Esta parte não faz parte do datapath reduzido. Serve apenas para comparar a versão por registradores com a formulação matricial.


In [7]:
D_direct = C.T @ np.array([
    [d00, d01, d02, d03],
    [d10, d11, d12, d13],
    [d20, d21, d22, d23],
    [d30, d31, d32, d33],
], dtype=object) @ C

S_direct = D_direct * G_reg
s_direct = A.T @ S_direct @ A

print("D_direct =")
print(D_direct)
print()
print("G_reg =")
print(G_reg)
print()
print("S_direct =")
print(S_direct)
print()
print("s_direct =")
print(s_direct)
print()
print("s_reg == s_direct?", np.array_equal(s_reg, s_direct))


D_direct =
[[Fraction(0, 1) Fraction(16, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(4, 1) Fraction(30, 1) Fraction(2, 1) Fraction(4, 1)]
 [Fraction(0, 1) Fraction(8, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(0, 1) Fraction(16, 1) Fraction(0, 1) Fraction(0, 1)]]

G_reg =
[[Fraction(0, 1) Fraction(-3, 2) Fraction(-1, 2) Fraction(-2, 1)]
 [Fraction(-9, 2) Fraction(9, 1) Fraction(3, 1) Fraction(15, 2)]
 [Fraction(-3, 2) Fraction(3, 1) Fraction(1, 1) Fraction(5, 2)]
 [Fraction(-6, 1) Fraction(21, 2) Fraction(7, 2) Fraction(8, 1)]]

S_direct =
[[Fraction(0, 1) Fraction(-24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(-18, 1) Fraction(270, 1) Fraction(6, 1) Fraction(30, 1)]
 [Fraction(0, 1) Fraction(24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(0, 1) Fraction(168, 1) Fraction(0, 1) Fraction(0, 1)]]

s_direct =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]

s_reg == s_direct? True


## 7. Contagem de registradores

### Datapath online, assumindo \(G\) pré-computado

- `d00..d33`: 16 registradores.
- `acc0..acc3`: 4 registradores.
- `y0,y1`: 2 registradores.
- `s00..s11`: 4 registradores.

Total:

$$
16 + 4 + 2 + 4 = 26
$$

Forma geral:

$$
K^2 + K + N + N^2
$$

com \(K=4\) e \(N=2\):

$$
4^2 + 4 + 2 + 2^2 = 26
$$

### Se \(G\) também for armazenado em registradores

Adicionar:

$$
K^2 = 16
$$

Total:

$$
26 + 16 = 42
$$

### Se a transformação de pesos também for feita online

Adicionar os registradores de \(g[3\times3]\) e, se necessário, os intermediários \(T[4\times3]\).  
Na prática, \(G\) costuma ser pré-computado offline para pesos fixos.


In [8]:
print("Contagem de registradores:")
print("d[4x4] = 16")
print("acc[4] = 4")
print("y[2] = 2")
print("s[2x2] = 4")
print("Total online sem G = 26")
print("Total online com G em registradores = 42")


Contagem de registradores:
d[4x4] = 16
acc[4] = 4
y[2] = 2
s[2x2] = 4
Total online sem G = 26
Total online com G em registradores = 42
